# Hakhel in the Septuagint


## Table of contents (ToC)<a class="anchor" id="TOC"></a>
* <a href="#bullet1">1 - Research question and limits</a>
* <a href="#bullet2">2 - Install the MCP server and notebook dependencies</a>
* <a href="#bullet3">3 - Load Perseus MCP</a>
* <a href="#bullet4">4 - Handle MCP responses</a>
* <a href="#bullet5">5 - Inspect the live Perseus MCP tool registry</a>
* <a href="#bullet6">6 - Verify the LXX scope and editions</a>
* <a href="#bullet7">7 - Retrieve Deuteronomy 31:10–13</a>
    * <a href="#bullet7x1">7.1 - The Greek audience formula</a>
* <a href="#bullet8">8 - Compare Deuteronomy 29:10–12</a>
* <a href="#bullet9">9 - Locate the Joshua parallel by its Greek formula</a>
* <a href="#bullet10">10 - Retrieve and compare the Joshua passage</a>
* <a href="#bullet11">11 - Search the selected books for the relevant Greek forms</a>
* <a href="#bullet12">12 - Some remarks on interpretation</a>
* <a href="#bullet13">13 - Export reproducible results</a>
* <a href="#bullet14">14 - Final check</a>
* <a href="#bullet15">15 - Required libraries</a>
* <a href="#bullet16">16 - Licences</a>
* <a href="#bullet17">17 - Notebook details</a>

# 1 - Research question and limits<a class="anchor" id="bullet1"></a>
##### [Back to ToC](#TOC)

The starting question is:

> How does the Septuagint represent the Hebrew טף in the two Deuteronomic covenant assemblies, and does the Joshua reading scene preserve the same Greek description?

Three distinctions are important:

1. **Hebrew observation** — The BHSA identifies the lexeme טף and its place in the Hebrew audience formula.
2. **Greek observation** — Perseus/Scaife supplies the wording of the selected Greek edition and the references returned by exact-form searches.
3. **Text-historical interpretation** — a judgement about translation technique, recensional history, or manuscript variants requires comparison with a critical edition and cannot be derived from a search count alone.

The notebook records the Greek wording and CTS identifiers before discussing their interpretation.

For AI assistance and researcher responsibility, see the [project disclosure](../README.md#bullet5).


# 2 - Install the MCP server and notebook dependencies<a class="anchor" id="bullet2"></a>
##### [Back to ToC](#TOC)

We will access the Septuagint text using [`perseus-mcp`](https://tonyjurg.github.io/Perseus-mcp/).

The `perseus-mcp` package is installed locally and opened through an in-process FastMCP client. Corpus metadata, searches, and passages are nevertheless retrieved through its specific [MCP tools](https://tonyjurg.github.io/Perseus-mcp/architecture/#tool-behaviour) from the Perseus/Scaife services.

Restart the kernel after installation if the imports fail in an already-running environment.

In [18]:
# Install the notebook dependencies, including MCP 1.30.0.
%pip install -q -r "requirements.txt"


Note: you may need to restart the kernel to use updated packages.


To check if the MCP server is installed, you can run the following cell:

In [19]:
import sys
# print(sys.executable)

%pip show perseus-mcp pandas mcp

Name: perseus-mcp
Version: 1.0.2
Summary: MCP server for Perseus Greek and Latin text research
Home-page: https://github.com/tonyjurg/Perseus-mcp
Author: Tony Jurg
Author-email: 
License: 
Location: c:\Users\tonyj\anaconda3\Lib\site-packages
Requires: defusedxml, fastmcp, httpx
Required-by: 
---
Name: pandas
Version: 2.2.3
Summary: Powerful data structures for data analysis, time series, and statistics
Home-page: https://pandas.pydata.org
Author: 
Author-email: The Pandas Development Team <pandas-dev@python.org>
License: BSD 3-Clause License

Copyright (c) 2008-2011, AQR Capital Management, LLC, Lambda Foundry, Inc. and PyData Development Team
All rights reserved.

Copyright (c) 2011-2023, Open source contributors.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are met:

* Redistributions of source code must retain the above copyright notice, this
  list of conditions and the following disclaimer.



# 3 - Load Perseus MCP<a class="anchor" id="bullet3"></a>
##### [Back to ToC](#TOC)

Start the kernel in `lxx` so the installation cell can find `requirements.txt`. The later path-discovery code also recognises the repository root.

Each run has a unique ID and a UTC start time recorded before source retrieval.
Each MCP request records its own start and end time, including failures.
The final export records completion time, retrieval dates, software versions, editions, and service endpoints.
These dates describe this session, including any cached responses; they are not publication dates.

The recorded run uses Tony Jurg, *Perseus MCP*, version 1.0.2 (Zenodo, 2026), [doi:10.5281/zenodo.20931139](https://doi.org/10.5281/zenodo.20931139).


In [20]:
# Load Perseus MCP
from __future__ import annotations

import importlib.metadata as package_metadata
import json
import logging
from pathlib import Path
from typing import Any

import pandas as pd
from fastmcp import Client
from perseus_mcp import server


def find_notebook_dir() -> Path:
    here = Path.cwd().resolve()
    if (here / "hakhel_lxx.ipynb").exists():
        return here
    if (here / "lxx" / "hakhel_lxx.ipynb").exists():
        return here / "lxx"
    raise FileNotFoundError("Start Jupyter in the repository root or in the lxx directory.")


NOTEBOOK_DIR = find_notebook_dir()
OUTPUT_DIR = NOTEBOOK_DIR / "lxx_hakhel_perseus_results"

mcp = server.mcp
logging.getLogger("fastmcp").setLevel(logging.CRITICAL)

print(f"Loaded MCP server: {mcp.name}")
# print(f"Output directory: {OUTPUT_DIR}") # Uncomment if you want the full path being shown

# Start recording before any source retrieval.
import sys
if str(NOTEBOOK_DIR.parent) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR.parent))
from notebook_provenance import RunProvenance
run_provenance = RunProvenance("lxx_core", OUTPUT_DIR, server)


Loaded MCP server: perseus


# 4 - Handle MCP responses<a class="anchor" id="bullet4"></a>
##### [Back to ToC](#TOC)

Perseus MCP returns text content blocks. Some blocks contain JSON, while the passage-text tool returns Greek as ordinary text. The functions below keep those response types separate.

In [21]:
# Handle MCP responses
def tool_text(result: Any) -> str:
    return "\n".join(
        block.text
        for block in result.content
        if getattr(block, "text", None) is not None
    )


async def call_text(client: Client, tool_name: str, arguments: dict[str, Any] | None = None) -> str:
    result = await run_provenance.call_tool(client, tool_name, arguments or {})
    return tool_text(result)


async def call_json(client: Client, tool_name: str, arguments: dict[str, Any] | None = None) -> Any:
    return json.loads(await call_text(client, tool_name, arguments))


def clean_text(value: str) -> str:
    return " ".join(str(value or "").split())


def search_urns(payload: dict[str, Any]) -> list[str]:
    return sorted(
        {
            row.get("passage", {}).get("urn")
            for row in payload.get("results", [])
            if row.get("passage", {}).get("urn")
        }
    )

### Optional: clear the local cache

This optional cell enforces a fresh read of the resources by clearing the local Perseus MCP metadata cache, both in memory and on disk.
Uncomment the code and run it before the retrieval cells when you want fresh metadata from the services.
The cache contains catalogue information and reference lists; passage text is retrieved separately.
This does not clear caches maintained by the remote services or remove the notebook’s saved results.
If you have already retrieved resources in this session, rerun the subsequent cells after clearing the cache.


In [22]:
# Uncomment the lines below to clear the local Perseus MCP metadata cache.
#async with Client(mcp) as client:
#     cache_result = await call_json(client, "clear_metadata_cache")
#print(cache_result)


# 5 - Inspect the live Perseus MCP tool registry<a class="anchor" id="bullet5"></a>
##### [Back to ToC](#TOC)

The notebook checks the running MCP server for the tools required by the analysis.

In [23]:
# Inspect the live Perseus MCP tool registry
async with Client(mcp) as client:
    tools = await client.list_tools()

tool_by_name = {tool.name: tool for tool in tools}
required_tools = {
    "get_scaife_library_metadata",
    "get_scaife_passage_json",
    "get_scaife_passage_text",
    "search_within_text",
}
missing = required_tools - set(tool_by_name)
assert not missing, f"Required Perseus MCP tools are missing: {sorted(missing)}"

tool_inventory = pd.DataFrame(
    [
        {
            "name": tool.name,
            "description": (tool.description or "").splitlines()[0],
        }
        for tool in tools
    ]
)
tool_inventory

,name,description
0,get_passage,Get the text of a specific passage using a CTS...
1,get_passage_plus,Get passage text plus surrounding metadata/con...
2,get_passage_plaintext,Get a passage as plain readable text instead o...
3,get_valid_references,"Get valid citations/references for a work, use..."
4,get_valid_references_json,Get valid citation references as paged JSON in...
5,count_valid_references,Count valid citation references without return...
6,get_capabilities,Get the list of available texts and editions f...
7,get_cache_status,Get local metadata cache status.
8,refresh_metadata_cache,Refresh cached CTS and Scaife library metadata.
9,clear_metadata_cache,Clear local metadata cache files and in-memory...


# 6 - Verify the LXX scope and editions<a class="anchor" id="bullet6"></a>
##### [Back to ToC](#TOC)

The Scaife library identifies the Greek Old Testament as `urn:cts:greekLit:tlg0527`, with Deuteronomy as work `tlg005` and Joshua as `tlg006`.
The code records these identifiers explicitly and checks that Scaife lists the selected Greek editions.

The [Scaife Library](https://scaife.perseus.org/library/) provides the catalogue records, including [Deuteronomy](https://scaife.perseus.org/library/urn:cts:greekLit:tlg0527.tlg005/).
The older Perseus CTS catalogue and the Scaife library do not list identical editions. Since this notebook retrieves Scaife passages, it checks their metadata through `get_scaife_library_metadata`.


In [24]:
# Verify the LXX scope and editions
LXX_TEXTGROUP = "urn:cts:greekLit:tlg0527"
DEUTERONOMY_WORK = f"{LXX_TEXTGROUP}.tlg005"
JOSHUA_WORK = f"{LXX_TEXTGROUP}.tlg006"
DEUTERONOMY_EDITION = f"{DEUTERONOMY_WORK}.1st1K-grc1"
JOSHUA_EDITION = f"{JOSHUA_WORK}.1st1K-grc1"

async with Client(mcp) as client:
    lxx_metadata = await call_json(client, "get_scaife_library_metadata", {"urn": LXX_TEXTGROUP})
    deuteronomy_metadata = await call_json(client, "get_scaife_library_metadata", {"urn": DEUTERONOMY_WORK})
    joshua_metadata = await call_json(client, "get_scaife_library_metadata", {"urn": JOSHUA_WORK})

deuteronomy_texts = {row["urn"] for row in deuteronomy_metadata.get("texts", [])}
joshua_texts = {row["urn"] for row in joshua_metadata.get("texts", [])}

assert DEUTERONOMY_EDITION in deuteronomy_texts
assert JOSHUA_EDITION in joshua_texts

edition_table = pd.DataFrame(
    [
        {"work": "Deuteronomy", "work_urn": DEUTERONOMY_WORK, "edition_urn": DEUTERONOMY_EDITION},
        {"work": "Joshua", "work_urn": JOSHUA_WORK, "edition_urn": JOSHUA_EDITION},
    ]
)
edition_table

,work,work_urn,edition_urn
0,Deuteronomy,urn:cts:greekLit:tlg0527.tlg005,urn:cts:greekLit:tlg0527.tlg005.1st1K-grc1
1,Joshua,urn:cts:greekLit:tlg0527.tlg006,urn:cts:greekLit:tlg0527.tlg006.1st1K-grc1


**Note:** An overview of the text under resource `tlg0527` can be found [here](https://atlas.perseus.tufts.edu/library/urn:cts:greekLit:tlg0527/).

# 7 - Retrieve Deuteronomy 31:10–13<a class="anchor" id="bullet7"></a>
##### [Back to ToC](#TOC)

We begin with the full Hakhel instruction rather than verse 12 alone. This keeps the seven-year setting, the Feast of Booths, the public reading, the audience list, and the pedagogical purpose chain together.

In [25]:
# Retrieve Deuteronomy 31:10–13
DEUTERONOMY_HAKHEL_URN = f"{DEUTERONOMY_EDITION}:31.10-31.13"

async with Client(mcp) as client:
    deuteronomy_hakhel_text = await call_text(
        client,
        "get_scaife_passage_text",
        {"urn": DEUTERONOMY_HAKHEL_URN},
    )
    deuteronomy_hakhel_json = await call_json(
        client,
        "get_scaife_passage_json",
        {"urn": DEUTERONOMY_HAKHEL_URN},
    )

print(clean_text(deuteronomy_hakhel_text))

καὶ ἐνετείλατο αὐτοῖς Μωυσῆς ἐν τῇ ἡμέρᾳ ἐκείνῃ λέγων Μετὰ ἑπτὰ ἔτη ἐν καιρῷ ἐνιαυτοῦ ἀφέσεως ἐν ἑορτῇ σκηνοπηγίας, ἐν τῷ συνπορεύεσθαι πάντα Ἰσραὴλ ὀφθῆναι ἐνώπιον κυρίου τοῦ θεοῦ ὑμῶν, ἐν τῶ τόπῳ ᾧ ἂν ἐκλέξηται κύριος, ἀναγνώσεσθε τὸν νόμον τοῦτον ἐναντίον παντὸς Ἰσραὴλ εἰς τὰ ὦτα αὐτῶν· ἐκκλησιάσας τὸν λαόν, τοὺς ἄνδρας καὶ τὰς γυναῖκας καὶ τὰ ἔκγονα καὶ τὸν προσήλυτον τὸν ἐν ταῖς πόλεσιν ὑμῶν, ἵνα ἀκούσωσιν καὶ ἵνα μάθωσιν φοβεῖσθαι Τύριον τὸν θεὸν ὑμῶν, καὶ ἀκούσονται ποιεῖν πάντας τοὺς λόγους τοῦ νόμου τούτου· καὶ οἱ υἱοὶ αὐτῶν οἳ οὐκ οἴδασιν ἀκούσονται, καὶ μαθήσονται φοβεῖσθαι κύριον τὸν θεόν σου πάσας τὰς ἡμέρας ὅσας αὐτοὶ ζῶσιν ἐπὶ τῆς γῆς εἰς ἢν ὑμεῖς διαβαίνετε τὸν Ἰορδάνην ἐκεῖ κληρονομεῖν αὐτήν.


**Historical observation on transcription:** Before October 27, 2026, the retrieved text contained `φοβεῖσθαι Τύριον τὸν θεὸν ὑμῶν` (Deut 31:12), which is clearly a transcription errors for κύριον. The retrieval by perseus-mcp was however faithfull to the original text on [scaife](https://scaife.perseus.org/reader/urn:cts:greekLit:tlg0527.tlg005.1st1K-grc1:31.12) and on the underlying [XML data](https://github.com/OpenGreekAndLatin/First1KGreek/blob/master/data/tlg0527/tlg005/tlg0527.tlg005.1st1K-grc1.xml#L5466). See for comparison, also the scan of the [printed version](https://archive.org/details/theoldtestamenti03swetuoft_202003/oldtestamentingr01swet/page/408/mode/2up). A data correction request was submitted under [issue 2891](https://github.com/OpenGreekAndLatin/First1KGreek/issues/2891). This resulted in [PR 2892](https://github.com/OpenGreekAndLatin/First1KGreek/pull/2892), committed on October 27, 2026 which corrected this specific OCR issue in 9 books in the LXX.


## 7.1 - The Greek audience formula<a class="anchor" id="bullet7x1"></a>
##### [Back to ToC](#TOC)

The central wording is:

> `ἐκκλησιάσας τὸν λαόν, τοὺς ἄνδρας καὶ τὰς γυναῖκας καὶ τὰ ἔκγονα καὶ τὸν προσήλυτον`

The Hebrew טף is represented by **τὰ ἔκγονα**, “the offspring” or “the descendants.” This is a collective description, but not a precise Greek age category. Verse 13 then speaks more specifically of `οἱ υἱοὶ αὐτῶν οἳ οὐκ οἴδασιν`, “their sons/children who do not know.” The same movement from a collective audience term to the next generation is therefore visible in Greek as well as in the Hebrew source.

The purpose clauses remain attached to the assembled group: they are to hear, learn, fear, and do. The Greek does not turn the offspring into a parenthetical group outside the audience.

In [26]:
# Check the Greek audience and purpose terms
deuteronomy_terms = [
    "ἐκκλησιάσας",
    "ἄνδρας",
    "γυναῖκας",
    "ἔκγονα",
    "προσήλυτον",
    "ἀκούσωσιν",
    "μάθωσιν",
    "φοβεῖσθαι",
    "υἱοὶ",
]

deuteronomy_term_check = pd.DataFrame(
    [
        {"term": term, "present": term in deuteronomy_hakhel_text}
        for term in deuteronomy_terms
    ]
)
deuteronomy_term_check

,term,present
0,ἐκκλησιάσας,True
1,ἄνδρας,True
2,γυναῖκας,True
3,ἔκγονα,True
4,προσήλυτον,True
5,ἀκούσωσιν,True
6,μάθωσιν,True
7,φοβεῖσθαι,True
8,υἱοὶ,True


# 8 - Compare Deuteronomy 29:10–12<a class="anchor" id="bullet8"></a>
##### [Back to ToC](#TOC)

Deuteronomy 29 provides the closest covenant-assembly comparison. We retrieve verses 10–12 in the numbering of the selected Greek edition.

Here too the audience includes the women, `τὰ ἔκγονα`, and the resident alien. The passage text returned by Perseus also contains an apparatus note recording `τέκνα` in Codex Alexandrinus at this point. That variant is worth recording, but its manuscript and text-critical significance must be checked in the critical edition itself. The notebook records the variant without attempting a manuscript comparison.

In [27]:
# Compare Deuteronomy 29:10–12
DEUTERONOMY_COVENANT_URN = f"{DEUTERONOMY_EDITION}:29.10-29.12"

async with Client(mcp) as client:
    deuteronomy_covenant_text = await call_text(
        client,
        "get_scaife_passage_text",
        {"urn": DEUTERONOMY_COVENANT_URN},
    )
    deuteronomy_covenant_json = await call_json(
        client,
        "get_scaife_passage_json",
        {"urn": DEUTERONOMY_COVENANT_URN},
    )

print(clean_text(deuteronomy_covenant_text))

Ὑμεῖς ἑστήκατε πάντες σήμερον ἐναντίον κυρίου τοῦ θεοῦ ὑμῶν, οἱ ἀρχίφυλοι ὑμῶν καὶ ἡ γερουσία ὑμῶν καὶ οἱ κριταὶ ὑμῶν καὶ οἱ γραμματοεισαγωγεῖς ὑμῶν, πᾶς ἀνὴρ Ἰσραήλ· αἱ γυναῖκες ὑμῶν καὶ τὰ ἔκγονα ὑμῶν καὶ ὁ προσήλυτος ὁ ἐν μέσῳ τῆς παρεμβολῆς ὑμῶν, ἀπὸ ξυλοπλοινοις 6 πλοιν|οις B* (πλοιοις B ab) | ἐν 2°] pr καὶ Α | om η B | u Α | πραθήσεσθαι Α XXIX 1 οὑς] ἦς Α | Μωση] τὼ 2 τοὺς υἵους] om του Α | τοι, θεράπουσιν] pr πάσι Α 3 εωρα- Α | ἐκείνα] + τὴν χεῖρα τὴν κραταιὰν καὶ ’τον βραχίονα ’τον ὑψηλὸν B ab mg A 4 ἔδωκ’ bis scr A* | βλέπειν sup ras (seq ras 2 litt) Aa pr του Α | ὤτα] pr τὰ Α 5 τεσσεράκοντα ἔτη ἐν τῆ ἐρήμω (ετη sup ras pr ras i fort litt) A | τεσσαρακ. Bb | ἐπαλαιώθησαν Α 6 κύριος] pr οὕτος Α | om ἐγὼ Α 7 βάσαν] pr τῆς Α | υμιν] ἥμιν Α | πολέμω] pr τὼ Α 8 Α | Μαννασση Α 9 ταυτης] + ποιεῖν αὐτοὺς Α | ποιήσητε Α Α | ἔναντι Α 11 αἰ γυναῖκες] pr καὶ Α | ἔκγονα] τέκνα Α B κόπου ὑμῶν καὶ ἕως ὑδροφόρου ὑμῶν, παρελθεῖν ἐν τῇ διαθήκῃ κυρίου τοῦ θεοῦ ὑμῶν καὶ ἐν ταῖς ἀρίς αὐτοῦ, ὅσα κύ

Or printed as a block:

In [28]:
import textwrap

print(textwrap.fill(clean_text(deuteronomy_covenant_text), width=100))

Ὑμεῖς ἑστήκατε πάντες σήμερον ἐναντίον κυρίου τοῦ θεοῦ ὑμῶν, οἱ ἀρχίφυλοι ὑμῶν καὶ ἡ γερουσία ὑμῶν
καὶ οἱ κριταὶ ὑμῶν καὶ οἱ γραμματοεισαγωγεῖς ὑμῶν, πᾶς ἀνὴρ Ἰσραήλ· αἱ γυναῖκες ὑμῶν καὶ τὰ ἔκγονα
ὑμῶν καὶ ὁ προσήλυτος ὁ ἐν μέσῳ τῆς παρεμβολῆς ὑμῶν, ἀπὸ ξυλοπλοινοις 6 πλοιν|οις B* (πλοιοις B ab)
| ἐν 2°] pr καὶ Α | om η B | u Α | πραθήσεσθαι Α XXIX 1 οὑς] ἦς Α | Μωση] τὼ 2 τοὺς υἵους] om του Α
| τοι, θεράπουσιν] pr πάσι Α 3 εωρα- Α | ἐκείνα] + τὴν χεῖρα τὴν κραταιὰν καὶ ’τον βραχίονα ’τον
ὑψηλὸν B ab mg A 4 ἔδωκ’ bis scr A* | βλέπειν sup ras (seq ras 2 litt) Aa pr του Α | ὤτα] pr τὰ Α 5
τεσσεράκοντα ἔτη ἐν τῆ ἐρήμω (ετη sup ras pr ras i fort litt) A | τεσσαρακ. Bb | ἐπαλαιώθησαν Α 6
κύριος] pr οὕτος Α | om ἐγὼ Α 7 βάσαν] pr τῆς Α | υμιν] ἥμιν Α | πολέμω] pr τὼ Α 8 Α | Μαννασση Α 9
ταυτης] + ποιεῖν αὐτοὺς Α | ποιήσητε Α Α | ἔναντι Α 11 αἰ γυναῖκες] pr καὶ Α | ἔκγονα] τέκνα Α B
κόπου ὑμῶν καὶ ἕως ὑδροφόρου ὑμῶν, παρελθεῖν ἐν τῇ διαθήκῃ κυρίου τοῦ θεοῦ ὑμῶν καὶ ἐν ταῖς ἀρίς
αὐτοῦ, ὅσα κύ

# 9 - Locate the Joshua parallel by its Greek formula<a class="anchor" id="bullet9"></a>
##### [Back to ToC](#TOC)

The Masoretic reference Josh 8:34–35 cannot simply be appended to the Greek edition URN. In the Scaife Joshua text, the covenant reading occurs at 9.7–9.8.

Rather than hard-coding that difference without evidence, we search Joshua for five words from the expected formula: reading, assembly, women, little children, and resident aliens. Their shared result identifies the passage.

In [29]:
# Locate the Joshua parallel by its Greek formula
JOSHUA_LOCATOR_FORMS = [
    "ἀνέγνω",
    "ἐκκλησίας",
    "γυναιξὶν",
    "παιδίοις",
    "προσηλύτοις",
]

joshua_locator_payloads = {}
async with Client(mcp) as client:
    for form in JOSHUA_LOCATOR_FORMS:
        joshua_locator_payloads[form] = await call_json(
            client,
            "search_within_text",
            {
                "query": form,
                "text_urn": JOSHUA_EDITION,
                "language": "greek",
                "query_format": "unicode",
                "search_kind": "form",
                "size": 50,
            },
        )

locator_rows = []
locator_sets = []
for form, payload in joshua_locator_payloads.items():
    urns = search_urns(payload)
    locator_sets.append(set(urns))
    locator_rows.append({"form": form, "count": len(urns), "passage_urns": urns})

joshua_locator_df = pd.DataFrame(locator_rows)
shared_joshua_urns = sorted(set.intersection(*locator_sets))

display(joshua_locator_df)
print("Shared result:", shared_joshua_urns)

,form,count,passage_urns
0,ἀνέγνω,2,[urn:cts:greekLit:tlg0527.tlg006.1st1K-grc1:9....
1,ἐκκλησίας,1,[urn:cts:greekLit:tlg0527.tlg006.1st1K-grc1:9.8]
2,γυναιξὶν,1,[urn:cts:greekLit:tlg0527.tlg006.1st1K-grc1:9.8]
3,παιδίοις,1,[urn:cts:greekLit:tlg0527.tlg006.1st1K-grc1:9.8]
4,προσηλύτοις,1,[urn:cts:greekLit:tlg0527.tlg006.1st1K-grc1:9.8]


Shared result: ['urn:cts:greekLit:tlg0527.tlg006.1st1K-grc1:9.8']


# 10 - Retrieve and compare the Joshua passage<a class="anchor" id="bullet10"></a>
##### [Back to ToC](#TOC)

The Greek Joshua formula is closely related to Deuteronomy but not lexically identical:

> `εἰς τὰ ὦτα πάσης ἐκκλησίας υἱῶν Ἰσραήλ, τοῖς ἀνδράσιν καὶ ταῖς γυναιξὶν καὶ τοῖς παιδίοις, καὶ τοῖς προσηλύτοις`

Where Deuteronomy has `τὰ ἔκγονα`, Joshua has `τοῖς παιδίοις`. Both can denote children, but `παιδίον` names a child or young person more directly, while `ἔκγονον` foregrounds descent or offspring. This difference shows why a single Greek equivalent cannot define Hebrew טף across these passages.

In [30]:
# Retrieve and compare the Joshua passage
JOSHUA_PARALLEL_URN = f"{JOSHUA_EDITION}:9.7-9.8"
JOSHUA_AUDIENCE_URN = f"{JOSHUA_EDITION}:9.8"

assert JOSHUA_AUDIENCE_URN in shared_joshua_urns

async with Client(mcp) as client:
    joshua_parallel_text = await call_text(
        client,
        "get_scaife_passage_text",
        {"urn": JOSHUA_PARALLEL_URN},
    )
    joshua_parallel_json = await call_json(
        client,
        "get_scaife_passage_json",
        {"urn": JOSHUA_PARALLEL_URN},
    )

print(textwrap.fill(clean_text(joshua_parallel_text), width=100))

# now build a nice table to compare; dynamicaly add the CTS passages.
comparison_df = pd.DataFrame(
    [
        {
            "Hebrew reference": "Deuteronomy 29:10 (BHSA/MT; Greek term at LXX 29.11)",
            "Greek CTS passage": DEUTERONOMY_COVENANT_URN,
            "Greek collective": "τὰ ἔκγονα",
            "immediate audience": "women + offspring + resident alien",
        },
        {
            "Hebrew reference": "Deuteronomy 31:12",
            "Greek CTS passage": DEUTERONOMY_HAKHEL_URN,
            "Greek collective": "τὰ ἔκγονα",
            "immediate audience": "men + women + offspring + resident alien",
        },
        {
            "Hebrew reference": "Joshua 8:35",
            "Greek CTS passage": JOSHUA_AUDIENCE_URN,
            "Greek collective": "τοῖς παιδίοις",
            "immediate audience": "men + women + children + resident aliens",
        },
    ]
)
comparison_df

καὶ μετὰ ταῦτα οὕτως ἀνέγνω Ἰησοῦς πάντα τὰ ῥήματα τοῦ νόμου τούτου, τὰς εὐλογίας καὶ τὰς κατάρας
κατὰ πάντα τὰ γεγραμμένα ἐν τῷ νόμῳ Μωυσῆ· οὐκ ἦν ἀπὸ πόντων ὧν ἐνετείλατο Μωυσῆς τῷ Ἰησοῖ ὃ οὐκ
ἀνέγνω εἰς τὰ ὦτα πάσης ἐκκλησίας υἱῶν Ἰσραήλ, τοῖς ἀνδράσιν καὶ ταῖς γυναιξὶν καὶ τοῖς παιδίοις,
καὶ τοῖς προσηλύτοις τοῖς προσπορευομένοις τῷ Ἰσραήλ. (X)


,Hebrew reference,Greek CTS passage,Greek collective,immediate audience
0,Deuteronomy 29:10 (BHSA/MT; Greek term at LXX ...,urn:cts:greekLit:tlg0527.tlg005.1st1K-grc1:29....,τὰ ἔκγονα,women + offspring + resident alien
1,Deuteronomy 31:12,urn:cts:greekLit:tlg0527.tlg005.1st1K-grc1:31....,τὰ ἔκγονα,men + women + offspring + resident alien
2,Joshua 8:35,urn:cts:greekLit:tlg0527.tlg006.1st1K-grc1:9.8,τοῖς παιδίοις,men + women + children + resident aliens


**Note:** the text for Josh 9:8 contains `οὐκ ἦν ἀπὸ πόντων`, this is a transcription errors for `οὐκ ἦν ἀπὸ πάντων`. The same data is found in the [source XML](https://github.com/OpenGreekAndLatin/First1KGreek/blob/master/data/tlg0527/tlg006/tlg0527.tlg006.1st1K-grc1.xml#L1406). The correct spelling is found in Henry Swete, [Septuagint, p437](https://archive.org/details/theoldtestamenti03swetuoft_202003/oldtestamentingr01swet/page/436/mode/2up).

# 11 - Search the selected books for the relevant Greek forms<a class="anchor" id="bullet11"></a>
##### [Back to ToC](#TOC)

We now run a small exact-form search within Deuteronomy and Joshua. This is not a Hebrew–Greek alignment: an occurrence of `ἔκγονα`, `τέκνα`, or `παιδία` may translate another Hebrew word, and an inflected form not listed here will not be counted. A lemma search would require a suitable index for the selected texts.

However, the current purpose is narrower. It is to show whether the forms used in the three central passages are isolated or participate in a broader vocabulary within each Greek book.

In [31]:
# Search the selected books for the relevant Greek forms
SEARCH_PLAN = [
    {"work": "Deuteronomy", "edition": DEUTERONOMY_EDITION, "form": "ἔκγονα"},
    {"work": "Deuteronomy", "edition": DEUTERONOMY_EDITION, "form": "τέκνα"},
    {"work": "Joshua", "edition": JOSHUA_EDITION, "form": "παιδία"},
    {"work": "Joshua", "edition": JOSHUA_EDITION, "form": "παιδίοις"},
    {"work": "Joshua", "edition": JOSHUA_EDITION, "form": "ἔκγονα"},
]

book_search_payloads = {}
book_search_rows = []
async with Client(mcp) as client:
    for item in SEARCH_PLAN:
        key = f"{item['work']}::{item['form']}"
        payload = await call_json(
            client,
            "search_within_text",
            {
                "query": item["form"],
                "text_urn": item["edition"],
                "language": "greek",
                "query_format": "unicode",
                "search_kind": "form",
                "size": 100,
            },
        )
        book_search_payloads[key] = payload
        urns = search_urns(payload)
        book_search_rows.append(
            {
                "work": item["work"],
                "form": item["form"],
                "exact_form_count": len(urns),
                "passage_urns": urns,
            }
        )

print('Results in selected books for the relevant Greek forms')

book_search_df = pd.DataFrame(book_search_rows)
book_search_df

Results in selected books for the relevant Greek forms


,work,form,exact_form_count,passage_urns
0,Deuteronomy,ἔκγονα,8,[urn:cts:greekLit:tlg0527.tlg005.1st1K-grc1:28...
1,Deuteronomy,τέκνα,10,[urn:cts:greekLit:tlg0527.tlg005.1st1K-grc1:11...
2,Joshua,παιδία,1,[urn:cts:greekLit:tlg0527.tlg006.1st1K-grc1:1.14]
3,Joshua,παιδίοις,1,[urn:cts:greekLit:tlg0527.tlg006.1st1K-grc1:9.8]
4,Joshua,ἔκγονα,0,[]


# 12 - Some remarks on interpretation<a class="anchor" id="bullet12"></a>
##### [Back to ToC](#TOC)

Several observations may now be made with reasonable confidence:

1. Deuteronomy 29 and 31 use the same Greek collective, `ἔκγονα`, in closely related covenant assemblies.
2. Deuteronomy 31:13 changes to `υἱοί` when it speaks of the next generation who 'do not yet know'.
3. The Joshua parallel preserves the inclusive audience formula but uses `παιδία` rather than `ἔκγονα`.
4. The LXX evidence therefore supports the inclusion of children or descendants without defining one exact age boundary.
5. The variation also means that searches for only `ἔκγονα` will miss a principal literary parallel.

Two cautions remain.
 - First, the retrieved Deut 29 text includes apparatus material and a variant `τέκνα`; manuscript claims should be checked against a critical edition.
 - Second, the numbering and arrangement of Joshua differ from the Masoretic text. The CTS URN is the reproducible identifier for the Greek passage, while “Josh 8:35” identifies its familiar Hebrew parallel.

# 13 - Export reproducible results<a class="anchor" id="bullet13"></a>
##### [Back to ToC](#TOC)

Next we can store the comparison table, exact-form search results, the three retrieved passages, the MCP tool inventory, and a small provenance file. Raw JSON responses are kept because they preserve the CTS metadata behind the compact tables.

In [32]:
# Export reproducible results
OUTPUT_DIR.mkdir(exist_ok=True)

comparison_df.to_csv(OUTPUT_DIR / "audiences.csv", index=False)
book_search_df.to_csv(OUTPUT_DIR / "form_searches.csv", index=False)
joshua_locator_df.to_csv(OUTPUT_DIR / "joshua_locator.csv", index=False)
tool_inventory.to_csv(OUTPUT_DIR / "mcp_tools.csv", index=False)

passage_rows = [
    {"label": "Deuteronomy 29:10-12", "urn": DEUTERONOMY_COVENANT_URN, "text": clean_text(deuteronomy_covenant_text)},
    {"label": "Deuteronomy 31:10-13", "urn": DEUTERONOMY_HAKHEL_URN, "text": clean_text(deuteronomy_hakhel_text)},
    {"label": "Joshua 9:7-8 (MT parallel Joshua 8:34-35)", "urn": JOSHUA_PARALLEL_URN, "text": clean_text(joshua_parallel_text)},
]
pd.DataFrame(passage_rows).to_csv(OUTPUT_DIR / "passages.csv", index=False)

raw_payload = {
    "metadata": {
        "lxx": lxx_metadata,
        "deuteronomy": deuteronomy_metadata,
        "joshua": joshua_metadata,
    },
    "passages": {
        DEUTERONOMY_COVENANT_URN: deuteronomy_covenant_json,
        DEUTERONOMY_HAKHEL_URN: deuteronomy_hakhel_json,
        JOSHUA_PARALLEL_URN: joshua_parallel_json,
    },
    "joshua_locator_searches": joshua_locator_payloads,
    "book_form_searches": book_search_payloads,
}
(OUTPUT_DIR / "lxx_raw_mcp_payloads.json").write_text(
    json.dumps(raw_payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

provenance = {
    **run_provenance.finish([DEUTERONOMY_EDITION, JOSHUA_EDITION]),
    "mcp_server": mcp.name,
    "perseus_mcp_version": package_metadata.version("perseus-mcp"),
    "fastmcp_version": package_metadata.version("fastmcp"),
    "lxx_textgroup": LXX_TEXTGROUP,
    "deuteronomy_edition": DEUTERONOMY_EDITION,
    "joshua_edition": JOSHUA_EDITION,
    "masoretic_to_greek_reference_note": "MT Joshua 8:34-35 corresponds to Greek Joshua 9:7-8 in this edition.",
}
print('Array of csv files created:')

(OUTPUT_DIR / "provenance.json").write_text(
    json.dumps(provenance, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

sorted(path.name for path in OUTPUT_DIR.iterdir())

Array of csv files created:


['audiences.csv',
 'form_searches.csv',
 'joshua_locator.csv',
 'lxx_raw_mcp_payloads.json',
 'mcp_tools.csv',
 'passages.csv',
 'provenance.json',
 'run_timing.json']

# 14 - Final check<a class="anchor" id="bullet14"></a>
##### [Back to ToC](#TOC)

The final assertions check the selected editions, passage references, and audience wording. A failed assertion stops the notebook and identifies a result that needs review.


In [33]:
# Final check
assert deuteronomy_term_check["present"].all()
assert "τὰ ἔκγονα" in deuteronomy_covenant_text
assert "τὰ ἔκγονα" in deuteronomy_hakhel_text
assert "τοῖς παιδίοις" in joshua_parallel_text
assert JOSHUA_AUDIENCE_URN in shared_joshua_urns
assert set(comparison_df["Greek collective"]) == {"τὰ ἔκγονα", "τοῖς παιδίοις"}
assert len(passage_rows) == 3

print("All LXX / Perseus MCP checks passed.")

All LXX / Perseus MCP checks passed.


# 15 - Required libraries<a class="anchor" id="bullet15"></a>
##### [Back to ToC](#TOC)

Install the dependencies from [requirements.txt](requirements.txt) in the environment used by this notebook:

```powershell
python -m pip install -r requirements.txt
```

Run this command from the notebook directory. The [reproduction guide](../REPRODUCING.md) describes environment creation and kernel selection.


# 16 - Licences<a class="anchor" id="bullet16"></a>
##### [Back to ToC](#TOC)

The original code in this notebook is licensed under the [MIT License](../LICENSE).
The original research writing and explanatory text are licensed under [Creative Commons Attribution 4.0 International (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/).
Credit Tony Jurg, link the licence and repository, and indicate any changes when reusing the original text; see the [text licence](../LICENSE-TEXT.md).

Quoted source material, retrieved passages, and datasets retain their own terms.
See [Sources, attribution, and reuse](../DATA_RIGHTS.md) for their licences and attribution requirements.


# 17 - Notebook details<a class="anchor" id="bullet17"></a>
##### [Back to ToC](#TOC)

<div style="float: left;">
  <table>
    <tr><td><strong>Author</strong></td><td>Tony Jurg</td></tr>
    <tr><td><strong>Version</strong></td><td>1.7</td></tr>
    <tr><td><strong>Date</strong></td><td>October 27, 2026</td></tr>
  </table>
</div>